# Transposable-element quantification near co-accessible enhancers

This tutorial identifies enhancers near a target genomic interval, intersects
them with transposable-element annotations, and writes TE-family and
TE-instance count matrices in 10x-compatible format.

## Requirements

Run this notebook from either the repository root or the `tutorials`
directory.

Required inputs:

- `example_data/chr2_coaccess_score_gt0.2.csv`
- `mm10.nrph.hits` or `mm10.nrph.hits.gz`, placed in the repository root or specified with the
  `TE_ANNOTATION` environment variable
- The `bedtools` command-line program
- The Python dependencies used by `TE_CRE_network.steamer_enhancer`

The co-accessibility coordinates and TE annotations must use the same genome
build. This example uses mm10 TE annotations.

In [ ]:
from contextlib import contextmanager
from pathlib import Path
import gzip
import os
import random
import shutil
import sys

from pybedtools import BedTool
from scipy.io import mmwrite
from scipy.sparse import csr_matrix


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)

    for candidate in (start, *start.parents):
        module = candidate / "TE_CRE_network" / "steamer_enhancer.py"
        if module.is_file():
            return candidate

    raise RuntimeError(
        "Could not locate the repository root containing "
        "TE_CRE_network/steamer_enhancer.py."
    )


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from TE_CRE_network import steamer_enhancer as se

In [ ]:
# User-configurable inputs
te_annotation_override = os.environ.get("TE_ANNOTATION")

if te_annotation_override:
    TE_ANNOTATION = Path(
        te_annotation_override
    ).expanduser().resolve()
else:
    annotation_candidates = [
        PROJECT_ROOT / "mm10.nrph.hits.gz",
        PROJECT_ROOT / "mm10.nrph.hits",
    ]

    TE_ANNOTATION = next(
        (
            candidate
            for candidate in annotation_candidates
            if candidate.is_file()
        ),
        annotation_candidates[0],
    )

COACCESSIBILITY_FILE = (
    PROJECT_ROOT
    / "example_data"
    / "chr2_coaccess_score_gt0.2.csv"
)

OUTPUT_DIR = PROJECT_ROOT / "results" / "TE_quant"

# Example target interval
TARGET_CHROMOSOME = "chr2"
TARGET_START = 140395308
TARGET_END = 140395595
RANDOM_SEED = 2017

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
required_files = {
    "TE annotation": TE_ANNOTATION,
    "co-accessibility data": COACCESSIBILITY_FILE,
}

missing = [
    label
    for label, path in required_files.items()
    if not path.is_file()
]

if missing:
    missing_text = ", ".join(missing)
    raise FileNotFoundError(
        f"Missing required input(s): {missing_text}. "
        "See the Requirements section above."
    )

if shutil.which("bedtools") is None:
    raise RuntimeError(
        "The bedtools executable was not found on PATH. "
        "Install bedtools before running this tutorial."
    )

if TARGET_START >= TARGET_END:
    raise ValueError("TARGET_START must be smaller than TARGET_END.")

In [ ]:
@contextmanager
def working_directory(path):
    previous_directory = Path.cwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(previous_directory)


# The current module writes TEs.bed, Frag.bed, and
# intersected_bed.bed into the working directory. Running these functions
# inside OUTPUT_DIR keeps all generated files in one controlled location.
random.seed(RANDOM_SEED)

enhancer_table = se.get_enhancers(str(COACCESSIBILITY_FILE))
enhancer_table = enhancer_table.drop_duplicates().reset_index(drop=True)

with working_directory(OUTPUT_DIR):
    se.create_bed_for_TEs(str(TE_ANNOTATION))

    nearby_enhancers, barcodes = se.get_nearby_enhancers(
        enhancer_table,
        TARGET_CHROMOSOME,
        TARGET_START,
        TARGET_END,
    )

    if len(barcodes) == 0:
        raise RuntimeError(
            "No enhancers were found near the requested target interval."
        )

    se.intersection(
        BedTool("TEs.bed"),
        BedTool("Frag.bed"),
    )

In [ ]:
te_bed_path = OUTPUT_DIR / "TEs.bed"
fragment_bed_path = OUTPUT_DIR / "Frag.bed"
intersection_path = OUTPUT_DIR / "intersected_bed.bed"

generated_bed_files = [
    te_bed_path,
    fragment_bed_path,
    intersection_path,
]

missing_outputs = [
    path.name for path in generated_bed_files if not path.is_file()
]

if missing_outputs:
    raise RuntimeError(
        "Expected BED output(s) were not created: "
        + ", ".join(missing_outputs)
    )

intersected_bed = BedTool(str(intersection_path))

(
    unique_te_table,
    te_family_table,
    unique_te_index,
    te_family_index,
    barcode_index,
) = se.make_cell_x_element_matrix(
    intersected_bed,
    barcodes,
)

if not unique_te_index:
    raise RuntimeError(
        "No overlaps were found between nearby enhancers and TE annotations."
    )

In [ ]:
def table_to_feature_by_barcode_matrix(
    table,
    feature_column,
    number_of_features,
    number_of_barcodes,
):
    return csr_matrix(
        (
            table["data"].to_numpy(),
            (
                table[feature_column].to_numpy(),
                table["barcode_index"].to_numpy(),
            ),
        ),
        shape=(number_of_features, number_of_barcodes),
    )


unique_te_matrix = table_to_feature_by_barcode_matrix(
    unique_te_table,
    feature_column="UniqueTE_index",
    number_of_features=len(unique_te_index),
    number_of_barcodes=len(barcode_index),
)

te_family_matrix = table_to_feature_by_barcode_matrix(
    te_family_table,
    feature_column="FamTE_index",
    number_of_features=len(te_family_index),
    number_of_barcodes=len(barcode_index),
)

In [ ]:
def ordered_names(index_mapping):
    return [
        name
        for name, _ in sorted(
            index_mapping.items(),
            key=lambda item: item[1],
        )
    ]


def write_lines_gzip(lines, output_path):
    with gzip.open(output_path, "wt", encoding="utf-8") as handle:
        for line in lines:
            handle.write(f"{line}\n")


def write_10x_matrix(matrix, feature_index, barcode_index, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)

    features = ordered_names(feature_index)
    barcodes = ordered_names(barcode_index)

    with gzip.open(
        output_dir / "features.tsv.gz",
        "wt",
        encoding="utf-8",
    ) as handle:
        for feature in features:
            handle.write(
                f"{feature}\t{feature}\tGene Expression\n"
            )

    write_lines_gzip(
        barcodes,
        output_dir / "barcodes.tsv.gz",
    )

    uncompressed_matrix = output_dir / "matrix.mtx"
    compressed_matrix = output_dir / "matrix.mtx.gz"

    mmwrite(str(uncompressed_matrix), matrix)

    with uncompressed_matrix.open("rb") as source:
        with gzip.open(compressed_matrix, "wb") as destination:
            shutil.copyfileobj(source, destination)

    uncompressed_matrix.unlink()


family_output_dir = OUTPUT_DIR / "TE_Fam_matrix"
unique_output_dir = OUTPUT_DIR / "TE_Unique_matrix"

write_10x_matrix(
    te_family_matrix,
    te_family_index,
    barcode_index,
    family_output_dir,
)

write_10x_matrix(
    unique_te_matrix,
    unique_te_index,
    barcode_index,
    unique_output_dir,
)

In [ ]:
print("TE quantification completed successfully.")
print(f"Nearby enhancers: {len(barcode_index):,}")
print(f"TE families: {te_family_matrix.shape[0]:,}")
print(f"Unique TE instances: {unique_te_matrix.shape[0]:,}")
print(f"TE-family matrix shape: {te_family_matrix.shape}")
print(f"Unique-TE matrix shape: {unique_te_matrix.shape}")
print("Outputs: results/TE_quant/")